In [1]:
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="Qwen/Qwen3-8B", base_url="https://api.siliconflow.cn")

from langchain.agents import create_agent, AgentState

## 敏感信息处理

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model,
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

# When user provides PII, it will be handled according to the strategy
result = agent.invoke({
    "messages": [{"role": "user", "content": "My email is john.doe@example.com and card is 5105-1051-0510-5100"}]
})
print(result)

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and card is ****-****-****-5100', additional_kwargs={}, response_metadata={}, id='e8de3a57-429d-433c-8b3e-a7f793f4f5e7'), AIMessage(content='It seems there might be a misunderstanding. I cannot access or retrieve your email address or card information. If you have any specific questions or need assistance with something else, feel free to let me know!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 35, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_name': 'Qwen/Qwen3-8B', 'system_fingerprint': '', 'id': '019b25b0b1ba8b18591efb6aa9d40898', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--7f906475-5cf4-43e1-8f63-2246f9a49553-0', usage_metadata={'input_tokens': 35, 'output

## human in the loop

In [8]:
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool


@tool
def search_tool(content: str) -> str:
    """搜索工具"""
    return "搜索的结果是true"


@tool
def send_email_tool(content: str, address: str) -> str:
    """发送邮件
        Args:
        content: 发送的内容
        address: 接收方的地址
    """
    return "发送成功"


@tool
def delete_database_tool(name: str) -> str:
    """删除数据库
    Args:
        name: 库名
    """
    return "删除成功"


agent = create_agent(
    model,
    tools=[search_tool, send_email_tool, delete_database_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # Require approval for sensitive operations
                "send_email_tool": True,
                "delete_database_tool": True,
                # Auto-approve safe operations
                "search_tool": False,
            }
        ),
    ],
    # Persist the state across interrupts
    checkpointer=InMemorySaver(),
)

# Human-in-the-loop requires a thread ID for persistence
config = {"configurable": {"thread_id": "1"}}

# Agent will pause and wait for approval before executing sensitive tools
result = agent.invoke(
    {"messages": [{"role": "user", "content": "发送邮件给mr lin，告诉他我今天不去喝酒了，地址是3027333@qq.com"}]},
    config=config
)

print(result['__interrupt__'])

result = agent.invoke(
    Command(resume={"decisions": [{"type": "reject","message": "不要使用我作为主语，使用小林"}]}),
    config=config  # Same thread ID to resume the paused conversation
)

[Interrupt(value={'action_requests': [{'name': 'send_email_tool', 'args': {'content': '我今天不去喝酒了', 'address': '3027333@qq.com'}, 'description': "Tool execution requires approval\n\nTool: send_email_tool\nArgs: {'content': '我今天不去喝酒了', 'address': '3027333@qq.com'}"}], 'review_configs': [{'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='fdb2d70fe3c335ef7a63319245b7a6fa')]
